In [ ]:
# ensures code compatibility and bundles GPU support
!pip install tensorflow[and-cuda]==2.15.0 tf-keras~=2.15.0 tensorrt-libs==8.6.1 --extra-index-url https://pypi.nvidia.com

In [ ]:
import tensorflow as tf
from tensorflow import keras

# check tensorflow version
print(tf.__version__)

# checks if GPU is available 
print("GPU available" if tf.config.list_physical_devices("GPU") else "not available")

In [ ]:
# Run this notebook using GPU
# Let's run the code first (until where we plot the graphs) before we discuss
# Let's start with the usual preliminaries

import tensorflow_datasets as tfds
import numpy as np

stopwords = ["a", "about", "above", "after", "again", "against", "all", "am", "an", "and", "any", "are", "as", "at",
             "be", "because", "been", "before", "being", "below", "between", "both", "but", "by", "could", "did", "do",
             "does", "doing", "down", "during", "each", "few", "for", "from", "further", "had", "has", "have", "having",
             "he", "hed", "hes", "her", "here", "heres", "hers", "herself", "him", "himself", "his", "how",
             "hows", "i", "id", "ill", "im", "ive", "if", "in", "into", "is", "it", "its", "itself",
             "lets", "me", "more", "most", "my", "myself", "nor", "of", "on", "once", "only", "or", "other", "ought",
             "our", "ours", "ourselves", "out", "over", "own", "same", "she", "shed", "shell", "shes", "should",
             "so", "some", "such", "than", "that", "thats", "the", "their", "theirs", "them", "themselves", "then",
             "there", "theres", "these", "they", "theyd", "theyll", "theyre", "theyve", "this", "those", "through",
             "to", "too", "under", "until", "up", "very", "was", "we", "wed", "well", "were", "weve", "were",
             "what", "whats", "when", "whens", "where", "wheres", "which", "while", "who", "whos", "whom", "why",
             "whys", "with", "would", "you", "youd", "youll", "youre", "youve", "your", "yours", "yourself",
             "yourselves"]

In [ ]:
# load in the training and test data
train_set = tfds.as_numpy(tfds.load('imdb_reviews', split="train"))
test_set = tfds.as_numpy(tfds.load('imdb_reviews', split = "test"))

In [ ]:
# we use a TextVectorization layer for cleaning and tokenization

from tensorflow.keras.layers import TextVectorization

import re
import string

def custom_standardization(input_data):
    # make all lowercase
    data = tf.strings.lower(input_data)
    # remove punctuation
    data = tf.strings.regex_replace(
        data, f"[{re.escape(string.punctuation)}]", ""
        )
    
    # remove HTML tags
    data = tf.strings.regex_replace(data, '<br />', ' ')
    
    # remove stopwords
    for i in stopwords:
        data = tf.strings.regex_replace(data, f" {i} ", " ")
    return data

# we can also define a customized split function
# if no "sep" is provided as the argument for the split function, we split according to whitespace by default
def custom_split(input_data):
    return tf.strings.split(input_data)

In [ ]:
# preparing integer sequences
max_length = 600
vocab_size = 20000

text_vectorization = TextVectorization(
    output_mode = 'int', 
    standardize = custom_standardization,
    split = custom_split, 
    # we set the vocab size to 20000 and max length of review to 600 words
    max_tokens = vocab_size,
    output_sequence_length = max_length
    )

In [ ]:
# create an empty list to store all the reviews and their labels later
train_imdb_reviews = []
train_labels = []
test_imdb_reviews = []
test_labels = []

for review in train_set:
    train_imdb_reviews.append(str(review['text']))
    train_labels.append(int(review['label']))
    
for review in test_set:
    test_imdb_reviews.append(str(review['text']))
    test_labels.append(int(review['label']))

In [ ]:
# out of the train set, we select 20% out for validation
validation_size = int(0.2 * len(train_imdb_reviews))
validation_size

In [ ]:
# create the actual train and validation set
actual_train_imdb_reviews = train_imdb_reviews[validation_size:]
val_imdb_reviews = train_imdb_reviews[0:validation_size]

actual_train_labels = train_labels[validation_size:]
val_labels = train_labels[0:validation_size]

len(actual_train_imdb_reviews)

In [ ]:
len(val_labels)

In [ ]:
# learn the vocabulary; remember that we should only be "seeing" the actual train dataset at this stage
text_vectorization.adapt(actual_train_imdb_reviews)

# subsequently we convert each review to a sequence of tokens, for train, val and test sets
vect_train_reviews = text_vectorization(actual_train_imdb_reviews)
vect_val_reviews = text_vectorization(val_imdb_reviews)
vect_test_reviews = text_vectorization(test_imdb_reviews)

# finally, we see the first review
vect_train_reviews[0]

In [ ]:
# Need to convert to numpy arrays for training

arr_train_reviews = np.array(vect_train_reviews)
arr_train_labels = np.array(actual_train_labels)

arr_val_reviews = np.array(vect_val_reviews)
arr_val_labels = np.array(val_labels)

arr_test_reviews = np.array(vect_test_reviews)
arr_test_labels = np.array(test_labels)

In [ ]:
import matplotlib.pyplot as plt

# we define a function for plotting the graphs, taking in the variable that holds the history of the training process
# and the metric of what we want to plot 
def plot_graphs(process, metric, filename):      
    plt.plot(process.history[metric])
    plt.plot(process.history['val_'+ metric])
    plt.title(filename.split('.')[0])
    plt.xlabel("Epochs")
    plt.ylabel(metric)
    plt.legend([metric, 'val_'+ metric])
    plt.savefig(filename)
    plt.show()


# Run the code above for each fine-tuning adjustment

In [ ]:
# Let's build our model 

# import the layers API from keras
from tensorflow.keras import layers

# we define the number of neurons in our hidden layer(s)
layer_dim = 16

# we also define the embedding dimension; setting this to 1024 means that each word (integer)
# will be "re-encoded" as a 1024-dimensional vector 

# It is common to see word embeddings that are 8-dimensional (for small datasets)
# up to 1024-dimensions when working with large datasets
# A higher dimensional embedding can capture fine-grained relationships between words
# but takes more data to learn
# we try the highest value possible first
embed_dim = 1024

# inputs as before, are 600-dimensional vectors
inputs = keras.Input(shape = (600,), dtype = 'int64')
# let's go to our slides to discuss 
x = layers.Embedding(input_dim = vocab_size, output_dim = embed_dim)(inputs)
# we use this to average all the embedding vectors in a review
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dense(layer_dim, activation = 'relu')(x)
outputs = layers.Dense(1, activation = 'sigmoid')(x)
model = keras.Model(inputs, outputs)

model.compile(optimizer = 'rmsprop',
                  loss = 'binary_crossentropy', 
                  metrics = ['accuracy'])

model.summary()


In [ ]:
# we train for 100 epochs
num_epochs = 100

# we specify our callback object
callbacks = [
    keras.callbacks.ModelCheckpoint('embeddings_model', save_best_only = True)
]

# we save the training history into a variable "original" so we can plot the results subsequently
original = model.fit(arr_train_reviews, arr_train_labels, epochs = num_epochs, 
         validation_data = (arr_val_reviews, arr_val_labels), callbacks = callbacks)

# we load our best model from training and apply it to the test dataset
model = keras.models.load_model('embeddings_model')
print(f"Test acc: {model.evaluate(arr_test_reviews, arr_test_labels)[1]:.3f}")

# What do we notice about the results? <go to slides>

In [ ]:
plot_graphs(original, "accuracy", "Original Accuracy.png")
plot_graphs(original, "loss", "Original Loss.png")

# For saved figures, look to the RHS panel, under the "Output" tab. Go to the ellipses for each 
# figure and choose "Download"
# Are the results optimal? Are we seeing overfitting? How might we improve the model? <slides>

# Improving model performance
## Reducing the learning rate

In [ ]:
# re-importing and re-building the model, so we don't have to re-run all the codes when re-opening notebook
from tensorflow.keras import layers

layer_dim = 16
embed_dim = 1024

inputs = keras.Input(shape = (None,), dtype = 'int64')
x = layers.Embedding(input_dim = vocab_size, output_dim = embed_dim)(inputs)
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dense(layer_dim, activation = 'relu')(x)
outputs = layers.Dense(1, activation = 'sigmoid')(x)
model = keras.Model(inputs, outputs)

# we adjust the learning rate, which has a default of 0.001
# we adjust it to 0.0001, 90% lower

'''
PROJECT: Try exploring values between 0.0001 and 0.001, or even beyond these values, to see 
how the performance of the model will be impacted
'''
# optimizer is the one that uses the learning rate
# we do the adjustment here, in our RMSprop object
RMS = keras.optimizers.RMSprop(learning_rate = 0.0001)

# pass in our RMSprop object when we configure our model for training
model.compile(optimizer = RMS,
                  loss = 'binary_crossentropy', 
                  metrics = ['accuracy'])

model.summary()

# train for 100 epochs
num_epochs = 100
# specify our callback object
callbacks = [
    keras.callbacks.ModelCheckpoint('low_learnrate_model', save_best_only = True)
]

# we save the training history into a variable so we can plot the results subsequently
change_learn_rate = model.fit(arr_train_reviews, arr_train_labels, epochs = num_epochs, 
         validation_data = (arr_val_reviews, arr_val_labels), callbacks = callbacks)

# we load our best model for applying to the test data
model = keras.models.load_model('low_learnrate_model')
print(f"Test acc: {model.evaluate(arr_test_reviews, arr_test_labels)[1]:.3f}")


In [ ]:
# Did the test accuracy improve? 

# Let's also check the plots again, and go back to the slides to discuss
plot_graphs(change_learn_rate, "accuracy", "Accuracy for learning rate = 0.0001.png")
plot_graphs(change_learn_rate, "loss", "Loss for learning rate = 0.0001.png")

# Trying other things - embedding dimension

In [ ]:
from tensorflow.keras import layers

layer_dim = 16

'''
PROJECT: Change this value to adjust the embedding dimension to another value, from 8 to 1024
besides the suggested value of 12 below
'''
adj_embed_dim = 12

inputs = keras.Input(shape = (None,), dtype = 'int64')
x = layers.Embedding(input_dim = vocab_size, output_dim = adj_embed_dim)(inputs)
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dense(layer_dim, activation = 'relu')(x)
outputs = layers.Dense(1, activation = 'sigmoid')(x)
model = keras.Model(inputs, outputs)


model.compile(optimizer = 'rmsprop',
                  loss = 'binary_crossentropy', 
                  metrics = ['accuracy'])

model.summary()

num_epochs = 100

callbacks = [
    keras.callbacks.ModelCheckpoint('adj_embeddings_model', save_best_only = True)
]


adj_embed = model.fit(arr_train_reviews, arr_train_labels, epochs = num_epochs, 
         validation_data = (arr_val_reviews, arr_val_labels), callbacks = callbacks)

model = keras.models.load_model('adj_embeddings_model')
print(f"Test acc: {model.evaluate(arr_test_reviews, arr_test_labels)[1]:.3f}")

In [ ]:
plot_graphs(adj_embed, "accuracy", "Adj Embed Accuracy.png")
plot_graphs(adj_embed, "loss", "Adj Embed Loss.png")

# Adjusting the model architecture

In [ ]:
from tensorflow.keras import layers

'''
PROJECT: Try adjusting new_layer_dim to adjust the number of neurons in the dense layer
'''
new_layer_dim = 8
adj_embed_dim = 1024

inputs = keras.Input(shape = (None,), dtype = 'int64')
x = layers.Embedding(input_dim = vocab_size, output_dim = adj_embed_dim)(inputs)
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dense(new_layer_dim, activation = 'relu')(x)

'''
PROJECT: We can also try increasing the number of layers
'''
# x = layers.Dense(new_layer_dim, activation = 'relu')(x)

outputs = layers.Dense(1, activation = 'sigmoid')(x)
model = keras.Model(inputs, outputs)


model.compile(optimizer = 'rmsprop',
                  loss = 'binary_crossentropy', 
                  metrics = ['accuracy'])

model.summary()

num_epochs = 100

callbacks = [
    keras.callbacks.ModelCheckpoint('new_dense_model', save_best_only = True)
]


new_dense = model.fit(arr_train_reviews, arr_train_labels, epochs = num_epochs, 
         validation_data = (arr_val_reviews, arr_val_labels), callbacks = callbacks)

model = keras.models.load_model('new_dense_model')
print(f"Test acc: {model.evaluate(arr_test_reviews, arr_test_labels)[1]:.3f}")


In [ ]:
plot_graphs(new_dense, "accuracy", "New Dense Accuracy.png")
plot_graphs(new_dense, "loss", "New Dense Loss.png")

# Using dropout

In [ ]:
from tensorflow.keras import layers

new_layer_dim = 16
adj_embed_dim = 1024

inputs = keras.Input(shape = (None,), dtype = 'int64')
x = layers.Embedding(input_dim = vocab_size, output_dim = adj_embed_dim)(inputs)
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dense(new_layer_dim, activation = 'relu')(x)

'''
PROJECT: Try experimenting with different percentages of neurons being dropped out
You don't have to use 16 nodes in the dense layer; 
You can try another value for the number of nodes in the dense layer
'''
x = layers.Dropout(0.5)(x)    
outputs = layers.Dense(1, activation = 'sigmoid')(x)
model = keras.Model(inputs, outputs)


model.compile(optimizer = 'rmsprop',
                  loss = 'binary_crossentropy', 
                  metrics = ['accuracy'])

model.summary()

num_epochs = 100

callbacks = [
    keras.callbacks.ModelCheckpoint('dropout_model', save_best_only = True)
]

dropout = model.fit(arr_train_reviews, arr_train_labels, epochs = num_epochs, 
         validation_data = (arr_val_reviews, arr_val_labels), callbacks = callbacks)

model = keras.models.load_model('dropout_model')
print(f"Test acc: {model.evaluate(arr_test_reviews, arr_test_labels)[1]:.3f}")

In [ ]:
plot_graphs(dropout, "accuracy", "Dropout Accuracy.png")
plot_graphs(dropout, "loss", "Dropout Loss.png")

# Using regularization

In [ ]:
from tensorflow.keras import layers

new_layer_dim = 16
adj_embed_dim = 1024

inputs = keras.Input(shape = (None,), dtype = 'int64')
x = layers.Embedding(input_dim = vocab_size, output_dim = adj_embed_dim)(inputs)
x = layers.GlobalAveragePooling1D()(x)

'''
PROJECT: Try adjust the regularization parameter of 0.01, with values from 0 to 0.1 
E.g. 0.1, 0.001, 0.0001 etc
'''
x = layers.Dense(new_layer_dim, activation = 'relu', 
                kernel_regularizer = keras.regularizers.l2(0.01))(x)

outputs = layers.Dense(1, activation = 'sigmoid')(x)
model = keras.Model(inputs, outputs)


model.compile(optimizer = 'rmsprop',
                  loss = 'binary_crossentropy', 
                  metrics = ['accuracy'])

model.summary()

num_epochs = 100

callbacks = [
    keras.callbacks.ModelCheckpoint('regularization_model', save_best_only = True)
]


regularization = model.fit(arr_train_reviews, arr_train_labels, epochs = num_epochs, 
         validation_data = (arr_val_reviews, arr_val_labels), callbacks = callbacks)

model = keras.models.load_model('regularization_model')
print(f"Test acc: {model.evaluate(arr_test_reviews, arr_test_labels)[1]:.3f}")


In [ ]:
plot_graphs(regularization, "accuracy", "Regularization Accuracy.png")
plot_graphs(regularization, "loss", "Regularization Loss.png")

# Finally, let's test our model on a new review 

In [ ]:
# Let's test our original embeddings model on a new review to see if it's able to classify it correctly
# You can try now using a new review from IMDB, or later during your project with your chosen model 
# here, we use a review of John Wick

new_review = ["Indiana jones, terminator, predator, Jurassic park, the matrix, Jason bourne, etc. All amazing franchises but all share a common trait of having a bad or forgettable movie or 2 at some point. This is why john wick is the best action action movie franchise of all time; The movies just keep getting better and better. In john wick 4, when the action starts in Osaka Japan, it never let's up despite it's almost 3 hour running time. What I really enjoyed the most about JW4 is the cast of characters. Great villains and allies really keep this movie interesting as john trots the globe for 3 hours. The sound engineering, editing, direction, cinematography are excellent as always. The color schemes throughout this franchise (more john wick 2, 3 and especially 4) are such a treat for the eyeballs. Best action franchise EVER. And the shotgun fight in Paris with the 'dragons breath' is the best action sequence I've probably ever seen in my life."]

# based on the review, do we think it is positive or negative? 

In [ ]:
# We vectorize the new review, by passing it to our text vectorization object
vect_new_review = text_vectorization(new_review)
vect_new_review

In [ ]:
# we convert numpy array
arr_vect_new_review = np.array(vect_new_review)
type(arr_vect_new_review)

In [ ]:
# we load the original embeddings model
model = keras.models.load_model('embeddings_model')

# we run predict on the new review and print the result
print(model.predict(arr_vect_new_review))

# Did the model come up with the correct prediction? 
# Let's go back to our slides to conclude our session